In [0]:

import uuid
import time
import traceback
from datetime import datetime, timezone

from pyspark.sql import functions as F
from delta.tables import DeltaTable

# ==========================================================
# CUSTOMER360 — GOLD VALIDATION + OPERATIONAL AUDIT
# ==========================================================

CATALOG = "dev_customer360"
AUDIT_TABLE = f"{CATALOG}.ops.pipeline_audit"
AUDIT_HISTORY_TABLE = f"{CATALOG}.ops.pipeline_audit_history"

# Unique identifier for this execution attempt.
attempt_id = str(uuid.uuid4())

TABLES = {
    "customers": f"{CATALOG}.gold.dim_customer_360",
    "orders": f"{CATALOG}.gold.fact_orders",
    "order_items": f"{CATALOG}.gold.fact_order_items",
    "revenue": f"{CATALOG}.gold.customer_revenue_summary",
    "rfm": f"{CATALOG}.gold.customer_rfm",
    "mapping": f"{CATALOG}.gold.customer_order_mapping"
}

KEYS = {
    "customers": ["customer_unique_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "revenue": ["customer_unique_id"],
    "rfm": ["customer_unique_id"],
    "mapping": ["customer_id"]
}

TASK_NAME = "validate_customer360_gold"
PIPELINE_NAME = "customer360_lakeflow_pipeline"


# ==========================================
# CONTROLLED FAILURE TEST CONFIGURATION
# ==========================================

try:
    dbutils.widgets.text("simulate_failure", "false")
except Exception:
    pass

SIMULATE_FAILURE = (
    dbutils.widgets.get("simulate_failure")
    .strip()
    .lower() == "true"
)

print("Failure simulation enabled:", SIMULATE_FAILURE)


# Job parameters are configured in the notebook task.
try:
    job_id = dbutils.widgets.get("audit_job_id") or None
except Exception:
    job_id = None

try:
    job_run_id = dbutils.widgets.get("audit_job_run_id") or None
except Exception:
    job_run_id = None

audit_id = (
    f"{job_run_id}:{TASK_NAME}"
    if job_run_id
    else str(uuid.uuid4())
)

started_at = datetime.now(timezone.utc)
start_clock = time.monotonic()

execution_status = "SUCCESS"
validation_passed = False
error_message = None
records_processed = None

# ----------------------------------------------------------
# Validation helper
# ----------------------------------------------------------

def check(condition, message):
    if not condition:
        raise ValueError(f"FAILED: {message}")

    print(f"PASSED: {message}")


def validate_gold():

    counts = {}

    # 1. Table existence and nonempty records
    for name, table in TABLES.items():
        count = spark.table(table).count()
        counts[name] = count

        check(count > 0, f"{name} contains records")
        print(f"{name}: {count:,} rows")

    # 2. Null and duplicate business keys
    for name, columns in KEYS.items():
        df = spark.table(TABLES[name])

        null_condition = None

        for column in columns:
            condition = F.col(column).isNull()

            null_condition = (
                condition
                if null_condition is None
                else null_condition | condition
            )

        null_keys = df.filter(null_condition).limit(1).count()

        check(null_keys == 0, f"{name}: no null keys")

        duplicate_keys = (
            df.groupBy(*columns)
              .count()
              .filter(F.col("count") > 1)
              .limit(1)
              .count()
        )

        check(
            duplicate_keys == 0,
            f"{name}: unique business keys"
        )

    # 3. Customer dataset consistency
    check(
        counts["customers"]
        == counts["revenue"]
        == counts["rfm"],
        "Customer, revenue and RFM counts match"
    )

    # 4. Payment reconciliation
    silver_total = (
        spark.table(f"{CATALOG}.silver.payments")
        .agg(F.sum("payment_value").alias("total"))
        .first()["total"]
    )

    gold_total = (
        spark.table(TABLES["orders"])
        .agg(F.sum("total_payment_value").alias("total"))
        .first()["total"]
    )

    check(
        silver_total is not None
        and gold_total is not None
        and silver_total == gold_total,
        "Silver payments equal Gold order payments"
    )

    print("Silver payment total:", silver_total)
    print("Gold payment total:", gold_total)

    # 5. Order mapping coverage
    unmapped_orders = (
        spark.table(TABLES["orders"]).alias("o")
        .join(
            spark.table(TABLES["mapping"]).alias("m"),
            F.col("o.customer_id") == F.col("m.customer_id"),
            "left_anti"
        )
        .limit(1)
        .count()
    )

    check(
        unmapped_orders == 0,
        "Every Gold order has a customer mapping"
    )

    print("=" * 50)
    print("CUSTOMER360 GOLD VALIDATION SUCCESSFUL")
    print("=" * 50)

    return sum(counts.values())


# ----------------------------------------------------------
# Execute validation
# ----------------------------------------------------------

validation_error = None


try:
    if SIMULATE_FAILURE:
        raise ValueError(
            "SIMULATED FAILURE: Gold validation recovery test"
        )

    records_processed = validate_gold()
    validation_passed = True


except Exception as exc:
    execution_status = "FAILED"
    error_message = "".join(
        traceback.format_exception_only(type(exc), exc)
    ).strip()[:4000]

    validation_error = exc
    print("GOLD VALIDATION FAILED:", error_message)


# ----------------------------------------------------------
# Write audit record (success or failure)
# ----------------------------------------------------------

ended_at = datetime.now(timezone.utc)

audit_values = [{
    "audit_id": audit_id,
    "job_id": job_id,
    "job_run_id": job_run_id,
    "task_name": TASK_NAME,
    "pipeline_name": PIPELINE_NAME,
    "execution_status": execution_status,
    "start_timestamp": started_at,
    "end_timestamp": ended_at,
    "duration_seconds": int(time.monotonic() - start_clock),
    "records_processed": records_processed,
    "records_rejected": None,
    "validation_passed": validation_passed,
    "error_message": error_message,
    "audit_timestamp": ended_at
}]

audit_df = spark.createDataFrame(
    [
        tuple(row[field.name] for field in spark.table(AUDIT_TABLE).schema)
        for row in audit_values
    ],
    schema=spark.table(AUDIT_TABLE).schema
)

audit_write_error = None

try:
    
    # Save this execution attempt permanently.
    # This runs for both successful and failed validations.
    history_df = audit_df.withColumn(
        "attempt_id",
        F.lit(attempt_id)
    ).select(
        *spark.table(AUDIT_HISTORY_TABLE).columns
    )

    history_df.write.mode("append").saveAsTable(
        AUDIT_HISTORY_TABLE
    )

    print(
        f"AUDIT HISTORY WRITTEN: {execution_status} | "
        f"Attempt ID: {attempt_id}"
    )


    (
        DeltaTable.forName(spark, AUDIT_TABLE)
        .alias("target")
        .merge(
            audit_df.alias("source"),
            "target.audit_id = source.audit_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"AUDIT WRITTEN: {execution_status} | "
        f"Run ID: {job_run_id or 'MANUAL'}"
    )

except Exception as exc:
    audit_write_error = exc
    print("AUDIT WRITE FAILED:", str(exc))


# ----------------------------------------------------------
# Preserve job failure behavior
# ----------------------------------------------------------

if validation_error is not None:
    raise RuntimeError(
        f"Customer360 Gold validation failed: {error_message}"
    ) from validation_error

if audit_write_error is not None:
    raise RuntimeError(
        "Gold validation passed, but audit persistence failed"
    ) from audit_write_error

print("CUSTOMER360 VALIDATION AND AUDIT COMPLETED")


Failure simulation enabled: False
PASSED: customers contains records
customers: 96,096 rows
PASSED: orders contains records
orders: 99,441 rows
PASSED: order_items contains records
order_items: 112,649 rows
PASSED: revenue contains records
revenue: 96,096 rows
PASSED: rfm contains records
rfm: 96,096 rows
PASSED: mapping contains records
mapping: 99,446 rows
PASSED: customers: no null keys
PASSED: customers: unique business keys
PASSED: orders: no null keys
PASSED: orders: unique business keys
PASSED: order_items: no null keys
PASSED: order_items: unique business keys
PASSED: revenue: no null keys
PASSED: revenue: unique business keys
PASSED: rfm: no null keys
PASSED: rfm: unique business keys
PASSED: mapping: no null keys
PASSED: mapping: unique business keys
PASSED: Customer, revenue and RFM counts match
PASSED: Silver payments equal Gold order payments
Silver payment total: 16008683.49
Gold payment total: 16008683.49
PASSED: Every Gold order has a customer mapping
CUSTOMER360 GOLD V

In [0]:
%sql
SELECT
    attempt_id,
    execution_status,
    validation_passed,
    error_message,
    audit_timestamp
FROM dev_customer360.ops.pipeline_audit_history
ORDER BY audit_timestamp DESC
LIMIT 5;



attempt_id,execution_status,validation_passed,error_message,audit_timestamp
1a4e4273-6670-4853-abcc-e9a6786145b3,SUCCESS,true,null,2026-10-14T11:21:04.760Z
f130aaef-4bc4-4043-baee-84f23a532d33,FAILED,false,ValueError: SIMULATED FAILURE: Gold validation recovery test,2026-10-14T11:19:57.960Z
37956943-f008-40eb-a0ed-e393c03ce3d9,FAILED,false,ValueError: SIMULATED FAILURE: Gold validation recovery test,2026-10-14T11:19:38.230Z
aa2ef0e8-11bc-4480-a6e3-a66b802d5e56,SUCCESS,true,null,2026-10-14T11:17:21.145Z
